# 02 — Boxes, angles, and rotated NMS

No model and no download: this lesson is pure geometry and runs anywhere in seconds. You will see why one rectangle has many five-number descriptions, how OrientedDet picks a canonical one (`le90`), how labels and augmentations produce boxes, and why duplicate removal must use rotated IoU.

| | |
|---|---|
| Needs | Only the install |
| Kaggle | Yes (CPU is fine) |
| Post | [Oriented-Det v0.1.0](https://deeplearning.earth/posts/2026-06-22_oriented-det_v0_1_0_sovereign_oriented_object_detection_for_eo/) |

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd))
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. One rectangle, many descriptions

A rectangle rotated by 180° is the same rectangle. So is the rectangle with width and height swapped and 90° added. `(w, h, θ)`, `(w, h, θ + π)`, and `(h, w, θ + π/2)` all draw identical pixels.

That matters for training. If labels mix these spellings, a network that predicts the right box in a different spelling is still penalised, and targets jump discontinuously near the angle limits. This is the **boundary problem** of oriented detection. The standard fix is a canonical form applied to every label and prediction. OrientedDet uses **le90**: `width` is the long side and `angle ∈ [-90°, 90°)`. `normalize_le90` maps any spelling to it.

In [ ]:
import math
from oriented_det.geometry import RBox, normalize_le90

spellings = {
    "original": RBox(200, 150, 160, 50, math.radians(30)),
    "+180 deg": RBox(200, 150, 160, 50, math.radians(210)),
    "w/h swapped, +90 deg": RBox(200, 150, 50, 160, math.radians(120)),
    "w/h swapped, -60 deg": RBox(200, 150, 50, 160, math.radians(-60)),
}
for name, b in spellings.items():
    n = normalize_le90(b)
    print(f"{name:22s} raw w={b.width:.0f} h={b.height:.0f} angle={math.degrees(b.angle):6.1f}"
          f"  ->  le90 w={n.width:.0f} h={n.height:.0f} angle={math.degrees(n.angle):5.1f}")

fig, ax = plt.subplots(figsize=(5, 4))
for (name, b), style in zip(spellings.items(), ["-", "--", ":", "-."]):
    xs, ys = zip(*b.corners(), b.corners()[0])
    ax.plot(xs, ys, style, lw=3, label=name)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.legend(fontsize=8); ax.set_title("four spellings, one rectangle")
plt.show()

## 2. From a labelled polygon to an `RBox`

Datasets rarely store `(cx, cy, w, h, θ)`. DOTA and most of its descendants store the **four corners** an annotator clicked, `x1 y1 x2 y2 x3 y3 x4 y4 class difficult`, and those quadrilaterals are rarely perfect rectangles. The loaders convert each one to the minimum-area rotated rectangle, then normalise it to le90.

In [ ]:
from oriented_det.data.dota import DOTAAnnotation

line = "112 40 230 110 205 152 88 83 large-vehicle 0"  # a slightly skewed hand-made quad
ann = DOTAAnnotation.from_line(line)
print(ann.class_name, "difficult" if ann.difficult else "", ann.rbox)

fig, ax = plt.subplots(figsize=(5, 3.5))
for pts, style, label in [(ann.polygon.points, "o-", "clicked corners"), (ann.rbox.corners(), "--", "fitted RBox")]:
    xs, ys = zip(*pts, pts[0])
    ax.plot(xs, ys, style, lw=2, label=label)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.legend(fontsize=8)
plt.show()

## 3. Augmentations move boxes too

Flipping or rotating an image during training must move every box with it, and the new angle must be normalised again. The DOTA recipes switch on horizontal, vertical, and diagonal flips (`preprocessing.enable_flip_*`); HRSC recipes add small random rotations. The functions in `oriented_det.geometry.transforms` do the box side:

In [ ]:
from oriented_det.geometry import transforms as T

W = H = 400
box = RBox(110, 90, 140, 44, math.radians(25))
moved = {
    "original": box,
    "flip_horizontal": T.flip_horizontal(box, W),
    "flip_vertical": T.flip_vertical(box, H),
    "rotate 90 deg about the centre": T.rotate(box, math.radians(90), W / 2, H / 2),
}
fig, axes = plt.subplots(1, len(moved), figsize=(16, 4))
for ax, (name, b) in zip(axes, moved.items()):
    n = normalize_le90(b)
    xs, ys = zip(*b.corners(), b.corners()[0])
    ax.fill(xs, ys, alpha=0.4); ax.plot(xs, ys)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect("equal")
    ax.set_title(f"{name}\nraw {math.degrees(b.angle):.0f} deg -> le90 {math.degrees(n.angle):.0f} deg", fontsize=9)
plt.tight_layout(); plt.show()

Note the flips and the rotation: the raw angle often comes back outside `[-90°, 90°)`. Training code calls `normalize_le90` after every transform for exactly this reason.

## 4. Rotated IoU vs axis-aligned IoU

Intersection-over-union is the overlap measure behind NMS, anchor matching, and mAP. Computed on AABBs it badly overestimates overlap between thin oriented objects. `rbox_iou` clips the real polygons.

In [ ]:
from oriented_det.ops import rbox_iou


def aabb_iou(a, b):
    a, b = a.axis_aligned_bounds(), b.axis_aligned_bounds()
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)


cases = {
    "crossing (X)": (RBox(100, 100, 160, 24, math.radians(45)), RBox(100, 100, 160, 24, math.radians(-45))),
    "side by side, 45 deg": (RBox(100, 100, 60, 24, math.radians(45)), RBox(121, 79, 60, 24, math.radians(45))),
    "same box": (RBox(100, 100, 60, 24, 0.3), RBox(100, 100, 60, 24, 0.3)),
}
for name, (a, b) in cases.items():
    print(f"{name:22s} rotated IoU {rbox_iou(a, b):.2f}   axis-aligned IoU {aabb_iou(a, b):.2f}")

## 5. Rotated NMS on a parking row

A detector fires several overlapping candidates per object. **Non-maximum suppression** keeps the highest-scoring box and deletes any other box whose IoU with it exceeds a threshold, then repeats. Below: six cars parked diagonally, each detected three times with a little jitter. Rotated NMS (`oriented_det.ops.rotated_nms`, the threshold 0.1 used by the DOTA checkpoints) is compared with ordinary NMS on the enclosing AABBs (`torchvision.ops.nms`).

In [ ]:
import torch
from torchvision.ops import nms as aabb_nms
from oriented_det.ops import rotated_nms

gen = torch.Generator().manual_seed(0)
rows = []
for k in range(6):  # 60x24 px cars at 45 deg, 32 px apart across their long side
    cx, cy = 90 + 23 * k, 210 - 23 * k
    for dup in range(3):
        jitter = torch.randn(3, generator=gen) * torch.tensor([1.5, 1.5, 0.04])
        rows.append([cx + jitter[0], cy + jitter[1], 60, 24, math.radians(45) + jitter[2], 0.9 - 0.2 * dup])
cand = torch.tensor(rows, dtype=torch.float32)
boxes, scores = cand[:, :5], cand[:, 5]

keep_rot = rotated_nms(boxes, scores, iou_threshold=0.1)
xyxy = torch.tensor([RBox(*b.tolist()).axis_aligned_bounds() for b in boxes], dtype=torch.float32)
keep_aabb = aabb_nms(xyxy, scores, iou_threshold=0.1)
print(f"{len(boxes)} candidates for 6 cars -> rotated NMS keeps {len(keep_rot)}, axis-aligned NMS keeps {len(keep_aabb)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx, title in [(axes[0], range(len(boxes)), "all candidates"),
                       (axes[1], keep_rot.tolist(), f"rotated NMS: {len(keep_rot)} kept"),
                       (axes[2], keep_aabb.tolist(), f"axis-aligned NMS: {len(keep_aabb)} kept")]:
    for i in idx:
        xs, ys = zip(*RBox(*boxes[i].tolist()).corners(), RBox(*boxes[i].tolist()).corners()[0])
        ax.plot(xs, ys, lw=1.5)
    ax.set_xlim(40, 260); ax.set_ylim(260, 40); ax.set_aspect("equal"); ax.set_title(title)
plt.tight_layout(); plt.show()

## What to try next

- Set the cars' angle to 0° in section 5: axis-aligned NMS now works too. The problem is orientation, not NMS itself.
- Raise the NMS threshold to 0.5 for both. AABB NMS recovers some cars, but in a dense harbour it also keeps real duplicates. That is why the DOTA checkpoints use rotated NMS at a tight 0.1.
- Apply `T.flip_diagonal(box, W, H)` (the third flip the DOTA recipes use) and other `T.rotate` angles to the box in section 3.
- **Next:** [03 — IoU, ProbIoU, and box losses](03_probiou_box_losses.ipynb): turning overlap into something a network can be trained on.